In [6]:
import sys
sys.path.append("../")
import numpy as np
import os
import torch
import torch.nn.parallel
import torch.optim
import torch.utils.data
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegressionCV
from dataset.vizdoom import VizdoomDataModule
from pytorch_lightning import Trainer
from pl_modules.comm import CoMM
from models.mmfusion import MMFusion
from models.input_adapters import PatchedInputAdapter, FeaturesInputAdapter
from models.alexnet import AlexNetEncoder
from models.vizdoom_encoder import VizdoomConvEncoder, VizdoomFFTEncoder
import warnings

In [4]:
torch.manual_seed(42) # for reproducibility
np.random.seed(42)
warnings.filterwarnings("ignore", category=UserWarning) # avoids sklearn warnings

In [8]:
comm = CoMM(
    encoder=MMFusion(
        encoders=[ # Symmetric visual encoders
            VizdoomConvEncoder(),
            VizdoomFFTEncoder()
        ],
        input_adapters=[ # Pach adapters for multimodal fusion
            FeaturesInputAdapter(n_features=512, dim_tokens=512),
            FeaturesInputAdapter(n_features=512, dim_tokens=512)
        ],
        embed_dim=512
    ),
    projection=CoMM._build_mlp(512, 512, 256),
    optim_kwargs=dict(lr=3e-4, weight_decay=1e-4),
    loss_kwargs=dict(temperature=0.1)
)

VizdoomConvEncoder: input_channels=3
Conv encoder output size: 512


In [10]:
state_dict = torch.load("../CoMM/bimodal_trifeatures/logs/version_37/checkpoints/epoch=99-step=50000.ckpt", weights_only=False)

In [11]:
state_dict.keys()

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters', 'hparams_type'])

In [15]:
comm.load_state_dict(state_dict['state_dict'])

<All keys matched successfully>

In [16]:
state_dict['state_dict']

OrderedDict([('encoder.input_adapters.0.weight',
              tensor([[-0.0870,  0.7887, -1.4379,  ..., -0.6400, -0.6362, -2.3458],
                      [ 1.1801, -0.7866, -1.2854,  ...,  1.0715, -1.6976, -1.4032],
                      [-0.9926, -0.5435, -0.9785,  ...,  0.6161, -2.0743, -0.2268],
                      ...,
                      [-1.3316, -0.5502,  0.1341,  ..., -0.5736,  0.6371, -0.0050],
                      [ 1.2293, -0.2725,  2.1078,  ..., -0.7423, -0.8167,  0.1169],
                      [ 0.0840, -0.0372, -2.0262,  ..., -1.0316, -0.3282,  0.4921]],
                     device='cuda:0')),
             ('encoder.input_adapters.0.bias',
              tensor([[ 1.1110,  1.0018,  0.4293,  ...,  0.3571, -0.0902, -1.2711],
                      [ 0.2170, -0.9212, -0.7706,  ...,  0.1051, -1.1952,  0.6721],
                      [-0.2605,  0.7465, -0.0211,  ..., -1.8168,  0.3028, -0.7075],
                      ...,
                      [-0.1064, -0.2987,  0.9340,  ..

In [17]:
data = torch.load("../datasets/my_way_home/test/data.pt")
data[0][0].shape

torch.Size([1, 3, 72, 128])

In [31]:
comm.encoder([data[0][0][0],data[1][0][None,:]], [True, True]).shape

RuntimeError: The size of tensor a (2) must match the size of tensor b (2520) at non-singleton dimension 2

In [24]:
data[1][0][0].shape

torch.Size([2520, 2])

In [33]:
data[0][0].shape

torch.Size([1, 3, 72, 128])

In [34]:
comm.encoder([data[0][0][0]], [True, False])

tensor([[-1.0370e+00,  1.6002e+00, -1.0181e-01, -1.3854e-01,  4.2385e-01,
          6.2872e-01,  8.8851e-01,  1.0926e+00, -5.1666e-01, -3.9292e-01,
         -1.3531e-01, -3.8187e-03, -3.1236e-01,  1.7363e-01,  2.7690e+00,
          1.3063e+00, -1.2143e+00,  6.7032e-01,  3.4656e-01, -1.0725e+00,
          1.0165e+00, -5.7164e-01, -2.0279e-02, -2.0766e+00,  3.9268e-01,
          1.6438e+00,  1.2784e+00,  7.1427e-01,  5.3748e-01, -7.6281e-01,
          5.8973e-01, -2.0916e-01,  5.4306e-01, -1.0243e+00,  7.9728e-01,
          3.4049e-01,  9.2702e-02, -4.1886e-01,  1.8702e-01, -1.6186e+00,
          2.4536e+00,  2.3373e+00, -8.7038e-01,  2.2235e+00, -7.9540e-01,
          4.6930e-01, -1.1864e+00,  3.0047e-02,  1.1623e+00, -4.2676e-01,
          8.4471e-01, -2.8705e+00, -5.9373e-01, -1.0697e+00, -1.0885e+00,
         -3.3491e-02,  7.0922e-01, -2.2056e+00, -7.9508e-01, -6.6019e-02,
         -3.8539e-01,  4.3352e-02, -4.1514e-01, -6.1277e-01, -8.4247e-01,
         -8.5364e-01,  3.0675e-01, -2.

In [39]:
comm.encoder([data[1][0],], [False, True]).shape

torch.Size([1, 512])

In [38]:
comm.encoder([data[0][0][0], data[1][0]]).shape

torch.Size([1, 512])

In [40]:
data[1][0]

tensor([[[0.4883, 0.4968],
         [0.4966, 0.5065],
         [0.5049, 0.5016],
         ...,
         [0.4943, 0.4952],
         [0.4915, 0.5016],
         [0.4882, 0.4935]]])